# 5.8 Loss 下降代表任務做得更好嗎？


希望模型寫出[1,2,3,4]。在正確前文下各格預測[1,2,0,4]，有三格對；自由生成卻接成[1,2,0,0]，只剩兩格對。兩份都沒有交出完整答案，因此逐格表現與整題完成要分開記。

訓練給每位置資料中的正確前文，叫teacher forcing；自由生成則把自己剛選的字接回去，第三格錯成0，就改變第四格的條件。先用這兩份手寫預測算分。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
truth = [1, 2, 3, 4]
teacher_forced = [1, 2, 0, 4]
generated = [1, 2, 0, 0]
for name, prediction in [("正確前文下", teacher_forced), ("自由生成", generated)]:
    correct = sum(a == b for a, b in zip(truth, prediction, strict=True))
    token_accuracy = correct / len(truth)
    exact_match = prediction == truth
    print(name, "逐字正確率", token_accuracy, "整串相同", exact_match)

輸入truth是真答案，兩份prediction分別模擬正確前文與自己前文的結果。`zip` 將同位置配對；`strict=True`要求兩份清單長度相同，否則提出ValueError並停止，不會默默略過較長清單的尾端。`a==b` 成立算1、不成立算0，`sum` 數正確位置；除以答案長4就得到token accuracy（逐片段正確率），這裡片段恰好一字。第一份三字正確，輸出0.75；第二份兩字正確，輸出0.5。

`prediction==truth` 比較完整清單，順序、長度與每格都相同才為True，這叫sequence exact match（整串完全匹配）。兩份都False。對於像算術答案、固定標籤這樣的任務，整串匹配常比逐字比例直接；對於開放式表達，合理答案可有多種寫法，完全匹配則不一定能反映好壞，評分方法需要按任務選擇。

loss量正確答案的機率，不只量最高候選是否選對；正確字機率0.6與0.9都可排名第一，代價仍不同。一筆已有故事模型接出「as a big was a she was a ber the」，雖開始用常見片段，尚未組成清楚句子。代價與實際回答應一起看。

還要確認題目提供了答案所需線索。只給顏色和形狀，沒給左右位置時，沒重現原文不能直接當成答錯已知事實。

練習只把 `teacher_forced` 的第三格0改成3。先預測第一份逐字正確率變1、整串相同True，第二份仍0.5和False，再執行核對。程式這次只改示意資料，沒有真的改變生成模型；要提升第二份表現，還得在實際生成過程中驗證訓練是否有效。

<details>
<summary>補充：實作約定與原始紀錄</summary>

再看一個真正訓練後的例子。[T.4的短文模型](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.4)在驗證資料的平均代價，從5.73454降到0.96282；可是給它開頭`color=blue;shape=circle;`，每次選最高分候選時，它接出`side=right.`。那篇驗證原文剩下的卻是`side=left.`。代價改善了，並沒有讓自由生成完整重現這篇原文。

這裡還有一個重要的判斷：提示只給顏色與形狀，沒有告訴模型物體在哪一邊；左或右都可能是合格式的續寫。所以這個差異能說明「沒有重現驗證原文」，不能單憑它宣判右邊是錯誤的事實。若任務真的要求判斷左右，就要在輸入提供相應線索，另定答案與成功條件。文字接續的loss、寫出合法格式、回答有真值的問題，各自需要相應的評估，這也是後面另做[屬性問答](https://birdhackor.github.io/tiny-perceptron-vlm/7.12.html)的理由。

換成真正的故事，這個差別更容易看到。[T.4的TinyStories實驗](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.4)用409篇故事訓練，在另外51篇驗證故事的全部42,453個計分位置，平均代價從5.76091降至1.80083。但是讓它每次選最高分候選、最多新增32個token（這份模型把文字拆成byte，也就是位元組；本例的英文字母與空格各占一byte，結束符號另占一個token，拆分方法會在[6.1](https://birdhackor.github.io/tiny-perceptron-vlm/6.1.html)展開），一筆實際結果是：

```text
給模型的開頭：Once upon a time there w
新生成的部分：as a big was a she was a ber the
```

它開始接出常見的英文片段，卻還沒有把片段組成清楚的句子，更沒有講出一個故事。這段輸出只是一筆觀察；平均代價用的是全部驗證文字，兩者不能互相取代。[完整實測報告](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/real_text.json)同時保留全部位置的分數與各留出側前八篇的生成示例，沒有因為續寫不好就省略。這也給我們一個實用的檢查順序：先確認模型確實學到一些文字規律，再問它能否在自己的前文下完成想要的任務。

短程式沿用本課工具與原計算語義。安裝、長訓練與重做操作見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，不需要先完成長配方才能閱讀這個例子。

</details>
